# CFD Forecasting Pipeline — Corrected

**Critical fixes from deep technical assessment (12 changes applied):**

| # | Fix | Before | After |
|---|-----|--------|-------|
| 1 | Nyquist Violation | pred_horizon=50 | pred_horizon=1 |
| 2 | POD Rank Bloat | r=150×3=450 | r=[30,50,30]=110 |
| 3 | Solid Masking | Enabled (0.11% cells) | Disabled |
| 4 | Inference Mode | Direct 50-step | Iterative rollout |
| 5 | Active Domain | Full mesh SVD | TKE>1% of max |
| 6 | Pressure Outliers | Silent clip | Diagnostic log |
| 7 | Model Size | 25.2M params | ~320K params |
| 8 | Loss | Fixed weights | Adaptive reweight |
| 9 | Training | 100 epochs | 500 + early stop |
| 10 | Metrics | RelErr only | Phase + amplitude + spectral |
| 11 | Validation | None | Pre-training checks |
| 12 | Logging | Minimal | Comprehensive |

**Target**: RelErr < 0.7 (was 1.1843 = FAILURE)


## [1/8] Imports & Configuration

In [ ]:
from __future__ import annotations
import os, gc, sys, time, warnings, copy, json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

# Import corrected pipeline modules
sys.path.insert(0, str(Path('.').resolve()))
from config_loader import load_config
from preprocessing import (
    diagnose_velocity_distribution,
    build_solid_mask,
    compute_tke_weights,
    shedding_frequency_diagnostic,
)
from pod_compression import (
    DEFAULT_POD_RANKS,
    diagnose_pressure_outliers,
    compute_pod_active_domain,
    compute_pod,
    pod_encode,
    pod_decode,
)
from model import make_model, count_params
from loss import AdaptiveWeightedLoss, compute_mode_weights
from training import train_model, PODLatentDataset
from inference import inference_iterative_rollout, reconstruct_from_latent
from metrics import ComprehensiveEvaluator, rel_err, rmse, mse
from validation_checks import PreTrainingValidator
from logger_utils import create_logger

print('All modules imported successfully.')


## [1/8] Configuration (Change 1, 2, 3, 7, 8, 9, 11)

In [ ]:
# ── Load corrected configuration from config.yaml ────────────────────────
try:
    import yaml
    with open('config.yaml') as f:
        CFG = yaml.safe_load(f)
    print('Loaded config.yaml')
except ImportError:
    print('PyYAML not available — using inline defaults')
    CFG = {}

# ── Runtime constants (edit these for your environment) ───────────────────
# Data paths — override with actual paths
HDF5_PATH   = CFG.get('data', {}).get('hdf5_path') or '/kaggle/input/cfd-case2/case2.h5'
COORDS_PATH = CFG.get('data', {}).get('coords_path') or '/kaggle/input/cfd-case2/coords.npy'
OUTPUT_DIR  = CFG.get('pipeline', {}).get('output_dir', 'outputs/case_2')
CHECKPOINT_DIR = 'checkpoints'
LOG_DIR = CFG.get('logging', {}).get('log_dir', 'logs')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# ── Split ─────────────────────────────────────────────────────────────────
TRAIN_STEPS   = CFG.get('split', {}).get('train_steps', 300)
PREDICT_STEPS = CFG.get('split', {}).get('predict_steps', 700)
TOTAL_STEPS   = TRAIN_STEPS + PREDICT_STEPS

# ── POD ranks (Change 2: reduced from 150 each) ───────────────────────────
POD_RANKS = CFG.get('pod', {}).get('ranks', DEFAULT_POD_RANKS)
LATENT_DIM = sum(POD_RANKS.values())  # 30+50+30 = 110
print(f'POD ranks: p={POD_RANKS["p"]}  u={POD_RANKS["u"]}  v={POD_RANKS["v"]}  '
      f'latent_dim={LATENT_DIM}  (was 450 with r=150 each)')

# ── Prediction horizon (Change 1: Nyquist fix) ────────────────────────────
PRED_HORIZON  = CFG.get('prediction', {}).get('pred_horizon', 1)  # FIXED: was 50
CONTEXT_LENGTH = CFG.get('prediction', {}).get('context_length', 150)
print(f'pred_horizon={PRED_HORIZON}  (was 50, violating Nyquist; shedding period=2 steps)')

# ── Model (Change 7: lightweight) ────────────────────────────────────────
MODEL_TYPE = CFG.get('model', {}).get('type', 'TransformerROM_Lite')
D_MODEL    = CFG.get('model', {}).get('d_model', 128)  # was 384
N_LAYERS   = CFG.get('model', {}).get('n_layers', 2)   # was 6
N_HEADS    = CFG.get('model', {}).get('n_heads', 8)
FF_MULT    = CFG.get('model', {}).get('ff_mult', 4)
DROPOUT    = CFG.get('model', {}).get('dropout', 0.1)

# ── Training (Change 9: extended + early stopping) ────────────────────────
EPOCHS       = CFG.get('training', {}).get('epochs', 500)  # was 100
BATCH_SIZE   = CFG.get('training', {}).get('batch_size', 32)
LR           = CFG.get('training', {}).get('lr', 3e-4)
LR_MIN       = CFG.get('training', {}).get('lr_min', 1e-6)
LR_SCHED     = CFG.get('training', {}).get('lr_scheduler', 'cosine_with_restarts')
T_0          = CFG.get('training', {}).get('T_0', 50)
WEIGHT_DECAY = CFG.get('training', {}).get('weight_decay', 1e-4)
GRAD_CLIP    = CFG.get('training', {}).get('grad_clip', 1.0)
USE_AMP      = CFG.get('training', {}).get('use_amp', True)
EARLY_STOP_CFG = CFG.get('training', {}).get('early_stopping', {
    'enabled': True, 'patience': 50, 'monitor': 'amplitude_loss', 'min_delta': 0.001
})

# ── Preprocessing (Change 3: solid masking disabled) ─────────────────────
SOLID_MASKING_ENABLED = CFG.get('preprocessing', {}).get('solid_masking', {}).get('enabled', False)

# ── Inference (Change 4: iterative rollout) ───────────────────────────────
INFERENCE_MODE = CFG.get('inference', {}).get('mode', 'iterative_rollout')

# ── Loss (Change 8: adaptive reweighting) ────────────────────────────────
WMSE_W      = CFG.get('loss', {}).get('weighted_mse_weight', 1.0)
AMP_W       = CFG.get('loss', {}).get('amplitude_weight', 0.5)
TKE_AMP_W   = CFG.get('loss', {}).get('tke_amplitude_weight', 0.5)
ADAPTIVE_CFG = CFG.get('loss', {}).get('adaptive_reweight', {
    'enabled': True, 'plateau_patience': 10, 'plateau_threshold': 0.01, 'amplitude_boost': 2.0
})

# ── Misc ─────────────────────────────────────────────────────────────────
DT         = CFG.get('data', {}).get('dt', 0.000222)  # time step seconds
SHEDDING_HZ = CFG.get('data', {}).get('shedding_freq_hz', 45.0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

print('\nConfiguration summary:')
print(f'  pred_horizon  : {PRED_HORIZON}  (Change 1)')
print(f'  latent_dim    : {LATENT_DIM}   (Change 2, was 450)')
print(f'  solid_masking : {SOLID_MASKING_ENABLED}  (Change 3)')
print(f'  model_type    : {MODEL_TYPE}  d_model={D_MODEL}  n_layers={N_LAYERS}')
print(f'  epochs        : {EPOCHS}  early_stop_patience={EARLY_STOP_CFG.get("patience",50)}')


## [2/8] Architecture Verification

In [ ]:
print('─' * 60)
print(f'  Architecture verification  d_model={D_MODEL}  n_layers={N_LAYERS}')
print(f'  latent_dim={LATENT_DIM}  pred_horizon={PRED_HORIZON}')
print('─' * 60)

models_to_check = {
    'LSTM': make_model('LSTM', latent_dim=LATENT_DIM, d_model=D_MODEL, n_layers=N_LAYERS, dropout=DROPOUT),
    'TransformerROM_Lite': make_model('TransformerROM_Lite', latent_dim=LATENT_DIM,
                                       d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                                       ff_mult=FF_MULT, dropout=DROPOUT, context_length=CONTEXT_LENGTH),
    'MambaLite': make_model('MambaLite', latent_dim=LATENT_DIM, d_model=D_MODEL, n_layers=N_LAYERS, dropout=DROPOUT),
}

for name, m in models_to_check.items():
    n = count_params(m)
    mb = n * 4 / 1e6
    print(f'  {name:<25}  params={n:>10,}  weights={mb:.1f} MB fp32')

print('─' * 60)
print(f'  Legacy Transformer (d=384,L=6):  params=25,213,506  (100.9 MB fp32)')
print(f'  Reduction: {25213506 // count_params(models_to_check["TransformerROM_Lite"])}× parameter reduction  (Change 7)')
print('─' * 60)


## [3/8] Load Data

In [ ]:
# ── Load mesh and field data ──────────────────────────────────────────────
# NOTE: Replace the mock data block below with real HDF5 loading for production use.
#
# Expected HDF5 layout:
#   /p  : (N_cells, T_total) float32 — pressure field
#   /u  : (N_cells, T_total) float32 — x-velocity field
#   /v  : (N_cells, T_total) float32 — y-velocity field
#   /coords : (N_cells, 2) float32   — spatial coordinates

USE_MOCK_DATA = not Path(HDF5_PATH).exists()
if USE_MOCK_DATA:
    print('[INFO] HDF5 not found — generating synthetic data for demonstration.')
    print('       Set HDF5_PATH and COORDS_PATH to real data files before production use.')
    N_CELLS = 2_113_842
    # Synthetic data: 2-step periodic oscillation (matches shedding_period=2 steps)
    np.random.seed(42)
    t = np.arange(TOTAL_STEPS)
    freq_steps = 2  # shedding period = 2 steps
    base_signal = np.sin(2 * np.pi * t / freq_steps)  # (T,)
    # Spatial modes: 30 POD modes for p, 50 for u, 30 for v
    def make_field(n_cells, T, n_modes, seed):
        rng = np.random.default_rng(seed)
        modes = rng.standard_normal((n_cells, n_modes)).astype(np.float32)
        norms = np.sqrt((modes ** 2).sum(axis=0, keepdims=True)) + 1e-8
        modes /= norms
        # Amplitude decays with mode index
        amps = np.exp(-0.1 * np.arange(n_modes)).astype(np.float32)
        # Temporal coefficients: harmonic + noise
        coeffs = np.zeros((T, n_modes), dtype=np.float32)
        for k in range(n_modes):
            phase = rng.uniform(0, 2 * np.pi)
            coeffs[:, k] = amps[k] * np.sin(2 * np.pi * t / freq_steps + phase).astype(np.float32)
        field = (modes @ coeffs.T).astype(np.float32)  # (N_cells, T)
        return field
    data_p = make_field(N_CELLS, TOTAL_STEPS, 30, seed=0)
    data_u = make_field(N_CELLS, TOTAL_STEPS, 50, seed=1)
    data_v = make_field(N_CELLS, TOTAL_STEPS, 30, seed=2)
    coords = np.random.default_rng(3).uniform(0, 1, (N_CELLS, 2)).astype(np.float32)
else:
    import h5py
    print(f'Loading {HDF5_PATH} ...')
    with h5py.File(HDF5_PATH, 'r') as f:
        data_p = f['p'][:].astype(np.float32)  # (N_cells, T)
        data_u = f['u'][:].astype(np.float32)
        data_v = f['v'][:].astype(np.float32)
    coords = np.load(COORDS_PATH).astype(np.float32)  # (N_cells, 2)
    N_CELLS = data_p.shape[0]

print(f'  N_cells={N_CELLS:,}  total steps={TOTAL_STEPS}')
assert data_p.shape == (N_CELLS, TOTAL_STEPS)
assert data_u.shape == (N_CELLS, TOTAL_STEPS)
assert data_v.shape == (N_CELLS, TOTAL_STEPS)


## [4/8] Preprocessing — Velocity Diagnostic & Solid Mask (Change 3)

In [ ]:
# ── Velocity distribution diagnostic ─────────────────────────────────────
u_train = data_u[:, :TRAIN_STEPS]
v_train = data_v[:, :TRAIN_STEPS]

if not USE_MOCK_DATA:
    vel_diag = diagnose_velocity_distribution(
        np.stack([u_train, v_train], axis=2),
        n_sample_steps=50,
    )

# ── Build fluid mask (Change 3: solid masking DISABLED) ───────────────────
vel_mean = np.sqrt(u_train ** 2 + v_train ** 2).mean(axis=1)  # (N_cells,)
fluid_mask = build_solid_mask(
    np.stack([u_train, v_train], axis=2),
    thresh=0.1,
    enabled=SOLID_MASKING_ENABLED,  # False by default (Change 3)
)
N_FLUID = int(fluid_mask.sum())
N_SOLID = N_CELLS - N_FLUID
print(f'  Fluid : {N_FLUID:,}')
print(f'  Solid : {N_SOLID:,}  ({N_SOLID / N_CELLS * 100:.2f}%)')

# ── TKE weights ───────────────────────────────────────────────────────────
tke_weights = compute_tke_weights(u_train, v_train, fluid_mask=fluid_mask)
print(f'  TKE weights: mean={tke_weights[fluid_mask].mean():.3f}  '
      f'max={tke_weights.max():.3f}  min={tke_weights[fluid_mask].min():.3f}')


## [5/8] POD Compression (Changes 2, 5, 6)

In [ ]:
# ── Active domain restriction (Change 5) ─────────────────────────────────
tke_active_fraction = CFG.get('pod', {}).get('active_domain', {}).get('tke_fraction', 0.01)
active_mask = compute_pod_active_domain(
    np.stack([u_train, v_train], axis=2),
    tke_fraction=tke_active_fraction,
)
# Intersect with fluid mask
active_mask = active_mask & fluid_mask
print(f'  Active + fluid cells: {active_mask.sum():,} / {N_CELLS:,}')

# ── POD compression with corrected ranks (Change 2) ──────────────────────
print(f'\n  Computing TKE weights from training velocity snapshots ...')
print(f'  POD ranks: p={POD_RANKS["p"]}  u={POD_RANKS["u"]}  v={POD_RANKS["v"]}  (was 150 each)')

pods = {}
coeff_list = []

for var, field_data, r in [
    ('p', data_p, POD_RANKS['p']),
    ('u', data_u, POD_RANKS['u']),
    ('v', data_v, POD_RANKS['v']),
]:
    X_train = field_data[:, :TRAIN_STEPS].astype(np.float64)

    # Pressure outlier diagnostics (Change 6)
    if var == 'p':
        outlier_pct = CFG.get('pod', {}).get('pressure_outlier_percentile', 2.5)
        lo, hi = diagnose_pressure_outliers(X_train, percentile=outlier_pct, label=var)
        X_train = np.clip(X_train, lo, hi)

    pod = compute_pod(
        X_train,
        r=r,
        tke_weights=tke_weights,
        active_mask=active_mask,
        label=var,
    )
    pods[var] = pod

    # Encode ALL timesteps (train + predict)
    X_all = field_data.astype(np.float64)
    if var == 'p':
        X_all = np.clip(X_all, lo, hi)
    A_all = pod_encode(X_all, pod)  # (T_total, r)
    coeff_list.append(A_all)

# ── Latent sequence ───────────────────────────────────────────────────────
Z_all = np.concatenate(coeff_list, axis=1).astype(np.float32)  # (T_total, D=110)
print(f'\n  Z: {Z_all.shape}  train={TRAIN_STEPS}  predict={PREDICT_STEPS}')
print(f'  Latent dim={LATENT_DIM} (was 450 — 75% reduction, Change 2)')

# Normalise Z by training statistics
Z_mean = Z_all[:TRAIN_STEPS].mean(axis=0, keepdims=True)
Z_std  = Z_all[:TRAIN_STEPS].std(axis=0, keepdims=True) + 1e-8
Z_norm = (Z_all - Z_mean) / Z_std  # (T_total, D)

Z_train_norm = Z_norm[:TRAIN_STEPS]
Z_val_norm   = Z_norm[max(0, TRAIN_STEPS - CONTEXT_LENGTH): TRAIN_STEPS]
Z_gt_norm    = Z_norm[TRAIN_STEPS:]   # ground-truth for evaluation

print(f'  Train windows: 0–{TRAIN_STEPS-1}')
print(f'  Val   windows: {TRAIN_STEPS - min(60, TRAIN_STEPS//5)}–{TRAIN_STEPS-1}')


## [5b/8] Shedding Frequency Diagnostic (Nyquist check)

In [ ]:
# ── Shedding frequency diagnostic ────────────────────────────────────────
for var in ('p', 'u'):
    A_var = coeff_list[['p','u','v'].index(var)][:TRAIN_STEPS, 0]  # mode 1 coeffs
    shedding_frequency_diagnostic(
        pod_coeffs=A_var,
        dt=DT,
        var_label=var,
        mode_idx=1,
        pred_horizon=PRED_HORIZON,  # Should be 1 now (Change 1)
    )


## [6/8] Pre-Training Configuration Validation (Change 11)

In [ ]:
# ── Run all pre-training checks (Change 11) ──────────────────────────────
validator = PreTrainingValidator(CFG)

# Collect POD energy data for validation
pod_energy_data = {}
for var in ('p', 'u', 'v'):
    pod = pods[var]
    pod_energy_data[var] = {
        'energy_fractions': pod['energy_fractions'],
        'r_used': pod['r_used'],
    }

# Build a demo model to count params
demo_model = make_model(MODEL_TYPE, latent_dim=LATENT_DIM, d_model=D_MODEL,
                         n_layers=N_LAYERS, n_heads=N_HEADS, ff_mult=FF_MULT,
                         dropout=DROPOUT, context_length=CONTEXT_LENGTH)
n_params = count_params(demo_model)
n_train_samples = len(Z_train_norm) - CONTEXT_LENGTH

# Shedding period in steps = 1 / (shedding_freq_hz * dt)
shedding_period_steps = 1.0 / (SHEDDING_HZ * DT + 1e-12)

n_passed, n_failed = validator.run_all(
    pred_horizon=PRED_HORIZON,
    shedding_period_steps=shedding_period_steps,
    pod_energy_data=pod_energy_data,
    n_model_params=n_params,
    n_train_samples=n_train_samples,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    latent_dim=LATENT_DIM,
    solid_masking_enabled=SOLID_MASKING_ENABLED,
    n_solid_cells=N_SOLID,
    n_total_cells=N_CELLS,
)

if n_failed > 0:
    print(f'WARNING: {n_failed} validation check(s) failed. Review before training.')
del demo_model


## [6b/8] Loss Function Setup (Changes 8, 10)

In [ ]:
# ── Mode weights from training variance ──────────────────────────────────
mode_weights = compute_mode_weights(Z_train_norm, floor=0.1)
tke_mode_weights = compute_mode_weights(Z_train_norm, floor=0.1)  # same base for demo

print('\n' + '─' * 60)
print('  Loss function verification')
print('─' * 60)
print(f'  mode_weights     : shape={mode_weights.shape}  mean={mode_weights.mean():.3f}  '
      f'w[0]={mode_weights[0]:.3f}  w[-1]={mode_weights[-1]:.3f}')
print(f'  tke_mode_weights : shape={tke_mode_weights.shape}  mean={tke_mode_weights.mean():.3f}  '
      f'max={tke_mode_weights.max():.3f}  min={tke_mode_weights.min():.3f}')

# Verify loss function behaviour
from loss import weighted_mse, amplitude_loss
import torch
t_dummy = torch.zeros(1, 20, LATENT_DIM)
p_dummy = torch.zeros(1, 20, LATENT_DIM)
assert float(weighted_mse(p_dummy, t_dummy, mode_weights)) == 0.0, 'wmse should be 0 on identical inputs'
print('  PASS  weighted_mse on identical inputs = 0')

# ── Create AdaptiveWeightedLoss (Change 8) ────────────────────────────────
criterion = AdaptiveWeightedLoss(
    mode_weights=mode_weights,
    tke_mode_weights=tke_mode_weights,
    wmse_weight=WMSE_W,
    amp_weight=AMP_W,
    tke_amp_weight=TKE_AMP_W,
    adaptive_enabled=ADAPTIVE_CFG.get('enabled', True),
    plateau_patience=ADAPTIVE_CFG.get('plateau_patience', 10),
    plateau_threshold=ADAPTIVE_CFG.get('plateau_threshold', 0.01),
    amplitude_boost=ADAPTIVE_CFG.get('amplitude_boost', 2.0),
)
print('─' * 60)


## [7/8] Training & Inference (Changes 4, 7, 9)

In [ ]:
print('=' * 60)
print(f'  {MODEL_TYPE}')
print('=' * 60)

# ── Create model (Change 7: TransformerROM_Lite) ──────────────────────────
model = make_model(
    MODEL_TYPE,
    latent_dim=LATENT_DIM,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    ff_mult=FF_MULT,
    dropout=DROPOUT,
    context_length=CONTEXT_LENGTH,
).to(DEVICE)

# Multi-GPU if available
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f'  DataParallel on {torch.cuda.device_count()} GPUs')

# ── Train (Change 9: 500 epochs + early stopping) ─────────────────────────
history = train_model(
    model=model,
    Z_train=Z_train_norm,
    Z_val=Z_val_norm,
    criterion=criterion,
    device=DEVICE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    lr_min=LR_MIN,
    lr_scheduler_type=LR_SCHED,
    T_0=T_0,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    context_length=CONTEXT_LENGTH,
    use_amp=USE_AMP and DEVICE.type == 'cuda',
    early_stopping_cfg=EARLY_STOP_CFG if EARLY_STOP_CFG.get('enabled', True) else None,
    checkpoint_dir=CHECKPOINT_DIR,
    model_name=MODEL_TYPE,
)

# ── Iterative one-step rollout inference (Change 4) ───────────────────────
print(f'\nInference: {PREDICT_STEPS} steps via {INFERENCE_MODE} ...')

actual_model = model.module if hasattr(model, 'module') else model
Z_pred_norm, inference_time = __import__('inference').run_inference_pipeline(
    model=actual_model,
    Z_all=Z_norm,
    train_steps=TRAIN_STEPS,
    predict_steps=PREDICT_STEPS,
    context_length=CONTEXT_LENGTH,
    device=DEVICE,
    use_amp=USE_AMP and DEVICE.type == 'cuda',
    mode=INFERENCE_MODE,
)
print(f'  Inference done  ({inference_time:.1f}s)')

# Denormalise predictions
Z_pred = Z_pred_norm * Z_std + Z_mean  # (T_pred, D)
Z_gt   = Z_all[TRAIN_STEPS:]           # (T_pred, D)  ground-truth


## [8/8] Evaluation — Comprehensive Metrics (Change 10)

In [ ]:
# ── Comprehensive evaluation (Change 10) ─────────────────────────────────
evaluator = ComprehensiveEvaluator(
    dt=DT,
    reference_freq_hz=SHEDDING_HZ,
    compute_phase=CFG.get('metrics', {}).get('compute_phase', True),
    compute_amplitude=CFG.get('metrics', {}).get('compute_amplitude', True),
    compute_spectral=CFG.get('metrics', {}).get('compute_spectral', True),
    compute_strouhal=CFG.get('metrics', {}).get('compute_strouhal', True),
    mode_subset=list(range(min(5, LATENT_DIM))),
)

metrics = evaluator.evaluate_latent(Z_pred, Z_gt)
evaluator.print_report(metrics, model_name=MODEL_TYPE)

# Summary table
print('=' * 60)
print(f'  Case 2  context={TRAIN_STEPS}  horizon={PREDICT_STEPS}')
print('=' * 60)
print(f'  {"Model":<25}  {"MSE":>12}  {"RMSE":>10}  {"RelErr":>8}')
print('  ' + '-' * 58)
print(f'  {MODEL_TYPE:<25}  {metrics["mse"]:>12.3e}  {metrics["rmse"]:>10.3e}  {metrics["rel_err"]:>8.4f}')
print('=' * 60)

if metrics['rel_err'] < 1.0:
    print(f'  RelErr = {metrics["rel_err"]:.4f} < 1.0 — BETTER than trivial mean predictor  ✓')
else:
    print(f'  RelErr = {metrics["rel_err"]:.4f} >= 1.0 — WORSE than trivial mean predictor  ✗')
    print(f'  (was 1.1843 before fixes; target < 0.7)')

if 'phase_error_deg' in metrics:
    print(f'  Phase error = {metrics["phase_error_deg"]:.1f}° (target < 15°)')
if 'amplitude_error' in metrics:
    print(f'  Amplitude error = {metrics["amplitude_error"]:.4f}  ratio={metrics.get("amplitude_ratio", float("nan")):.3f}')


## Experiment Logging & Save (Change 12)

In [ ]:
# ── Comprehensive experiment logging (Change 12) ─────────────────────────
logger = create_logger(CFG, log_dir=LOG_DIR)

# Log POD truncation
for var in ('p', 'u', 'v'):
    pod = pods[var]
    r = pod['r_used']
    ef = pod['energy_fractions']
    logger.log_pod_truncation(
        var=var,
        r_used=r,
        r90=pod['r90'],
        r99=pod['r99'],
        r999=pod['r999'],
        cum_energy_at_r=float(ef[r-1]) if r <= len(ef) else float(ef[-1]),
        n_active_cells=int(active_mask.sum()),
        n_total_cells=N_CELLS,
    )
    logger.log_energy_summary(var, pod['sigma'], r, float(pod['sigma'].sum()))

logger.save_truncation_log()
logger.save_energy_summary()

# Log mode weights
logger.log_mode_weights(mode_weights.numpy(), tke_mode_weights.numpy())

# Log training history
for i, (tl, vl) in enumerate(zip(history['train_loss'], history['val_loss'])):
    logger.log_training_epoch(
        epoch=i+1,
        model_name=MODEL_TYPE,
        train_loss=tl,
        val_loss=vl,
        lr=history['lr'][i] if i < len(history['lr']) else 0.0,
        amp_weight=criterion.amp_weight,
    )
logger.save_training_history()

# Log evaluation
logger.log_evaluation(
    model_name=MODEL_TYPE,
    metrics=metrics,
    pred_horizon=PRED_HORIZON,
    inference_mode=INFERENCE_MODE,
)
logger.save_evaluation()
logger.print_final_summary()

print(f'\nOutput directory: {OUTPUT_DIR}/')


## Prediction Visualisation

In [ ]:
# ── Plot predicted vs ground-truth POD coefficients ─────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 10), constrained_layout=True)
fig.suptitle(f'{MODEL_TYPE} — Predicted vs Ground-Truth POD Coefficients\n'
             f'(latent_dim={LATENT_DIM}, pred_horizon={PRED_HORIZON}, '
             f'RelErr={metrics["rel_err"]:.4f})', fontsize=13)

t_pred = np.arange(PREDICT_STEPS)
var_offsets = {'p': 0, 'u': POD_RANKS['p'], 'v': POD_RANKS['p'] + POD_RANKS['u']}

for row, var in enumerate(('p', 'u', 'v')):
    offset = var_offsets[var]
    for col in range(5):
        ax = axes[row, col]
        mode_idx = offset + col
        if mode_idx < LATENT_DIM:
            ax.plot(t_pred, Z_gt[:, mode_idx], 'b-', linewidth=1.0, label='GT', alpha=0.8)
            ax.plot(t_pred, Z_pred[:, mode_idx], 'r--', linewidth=1.0, label='Pred', alpha=0.8)
            ax.set_title(f'{var} mode {col+1}', fontsize=9)
            ax.tick_params(labelsize=7)
        if row == 0 and col == 0:
            ax.legend(fontsize=7)

plt.savefig(f'{OUTPUT_DIR}/pred_vs_gt_{MODEL_TYPE}.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'  Saved pred_vs_gt_{MODEL_TYPE}.png')

# ── Plot training loss curve ──────────────────────────────────────────────
if history['train_loss']:
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    epochs_arr = np.arange(1, len(history['train_loss']) + 1)
    ax.semilogy(epochs_arr, history['train_loss'], label='Train loss')
    ax.semilogy(epochs_arr, history['val_loss'],   label='Val loss')
    if history.get('train_amp'):
        ax.semilogy(epochs_arr, history['train_amp'], label='Amp loss', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
    ax.set_title(f'{MODEL_TYPE} training curve')
    ax.legend()
    plt.savefig(f'{OUTPUT_DIR}/loss_{MODEL_TYPE}.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  Saved loss_{MODEL_TYPE}.png')
